## 🎯 Learning Objectives
* Understand the critical role of evaluating answer quality in agentic RAG systems, especially for eCommerce applications.
* Learn what constitutes a 'golden dataset' and its importance in establishing ground truth for RAG evaluation.
* Identify key metrics for RAG answer quality, including faithfulness, relevancy, context recall, and context precision.
* Implement a practical evaluation workflow using a modern RAG evaluation framework like Ragas.
* Interpret evaluation results to diagnose common RAG issues such as hallucination, irrelevance, and poor retrieval.


## Evaluating Answer Quality Against a Golden Dataset

In the realm of agentic RAG (Retrieval Augmented Generation) systems, particularly for high-stakes applications like eCommerce, the quality of generated answers is paramount. An incorrect product recommendation, a hallucinated feature, or a misleading price can directly impact customer trust and business revenue. This lesson focuses on how to systematically evaluate the quality of answers produced by your agentic RAG system using a **golden dataset**.

### What is a Golden Dataset?

Imagine you're a teacher grading an exam. To ensure fairness and accuracy, you don't just guess the right answers; you use an **answer key**. In the world of AI, a golden dataset serves as this answer key. It's a meticulously curated collection of queries, their corresponding *ground truth answers*, and often the *ground truth contexts* that should have been retrieved to formulate those answers. This dataset represents the 'ideal' output for a given input, providing an objective benchmark against which your RAG system's performance can be measured.

For an eCommerce agent, a golden dataset entry might look like this:

*   **Query:** "What are the key features of the 'EcoCharge Pro' wireless charger?"
*   **Ground Truth Answer:** "The EcoCharge Pro features 15W fast charging, multi-device compatibility (phones, smartwatches, earbuds), a sleek recycled aluminum design, and intelligent temperature control for safe charging."
*   **Ground Truth Contexts:** ["EcoCharge Pro Product Page - Features", "EcoCharge Pro User Manual - Specifications"]

### Why is Evaluation Crucial for Agentic RAG?

Agentic RAG systems, by their nature, involve multiple steps (query understanding, retrieval, generation, potentially self-correction). Each step introduces potential points of failure. Without robust evaluation:

1.  **Hallucinations go undetected:** The agent might confidently generate plausible but incorrect information.
2.  **Irrelevant answers persist:** The system might provide technically correct but unhelpful responses.
3.  **Contextual gaps remain:** The retrieval mechanism might consistently miss crucial information.
4.  **Performance regressions occur:** Updates to models or retrieval strategies could degrade quality without notice.

### Key Metrics for RAG Evaluation

Modern RAG evaluation frameworks leverage Large Language Models (LLMs) as 'judges' to assess various aspects of answer quality. Some core metrics include:

*   **Faithfulness:** Does the generated answer contain information that is directly supported by the retrieved context? (Detects hallucination).
*   **Answer Relevancy:** Is the generated answer directly relevant to the user's query? (Detects off-topic responses).
*   **Context Recall:** Does the retrieved context contain all the necessary information to answer the query? (Detects missing information in retrieval).
*   **Context Precision:** Is all the retrieved context relevant to the query? (Detects noisy or irrelevant retrieved documents).
*   **Answer Similarity:** How semantically similar is the generated answer to the ground truth answer? (Measures correctness against a known good answer).

By combining these metrics, we gain a comprehensive understanding of our RAG system's strengths and weaknesses, enabling targeted improvements. In the following code example, we'll demonstrate how to set up a basic evaluation using a golden dataset and a popular RAG evaluation framework.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install ragas datasets openai tiktoken

import os
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision, answer_similarity

# --- Configuration for LLM-as-a-Judge --- 
# Ragas uses LLMs to evaluate metrics. For 2026, we assume access to powerful models.
# Set your API keys as environment variables or directly here (not recommended for production).
# Example for OpenAI:
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# Example for Hugging Face (for some models or embeddings):
# os.environ["HUGGINGFACE_API_KEY"] = "YOUR_HUGGINGFACE_API_KEY"

# For demonstration, we'll use a placeholder for API key check.
# In a real scenario, ensure these are set.
if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY environment variable not set. Ragas evaluation might fail or use default models.")
    print("Please set it for full functionality, e.g., `export OPENAI_API_KEY='sk-...'`")

# --- 1. Define a Golden Dataset (Ground Truth) ---
# In a real-world scenario, this would be a much larger, carefully curated dataset.
# Each entry includes the query, the ideal answer, and the ideal context(s).

golden_data = [
    {
        "question": "What are the main features of the 'Quantum Leap' gaming laptop?",
        "ground_truth_answer": "The Quantum Leap gaming laptop boasts an Intel Core i9-14900HX processor, NVIDIA GeForce RTX 5090 GPU, 32GB DDR5 RAM, a 17-inch QHD 240Hz display, and a 2TB NVMe SSD.",
        "ground_truth_contexts": [
            "Quantum Leap Product Page: Features - Intel Core i9-14900HX, NVIDIA GeForce RTX 5090, 32GB DDR5, 17-inch QHD 240Hz, 2TB NVMe SSD."
        ]
    },
    {
        "question": "How do I troubleshoot connectivity issues with the 'AuraFlow' smart air purifier?",
        "ground_truth_answer": "To troubleshoot AuraFlow connectivity, first ensure it's plugged in and within Wi-Fi range. Try restarting the device and your router. If issues persist, reset the purifier to factory settings via the mobile app and re-pair it.",
        "ground_truth_contexts": [
            "AuraFlow User Manual: Troubleshooting - Connectivity Issues",
            "AuraFlow Support FAQ: Wi-Fi Setup Guide"
        ]
    },
    {
        "question": "What is the return policy for electronics purchased from AgenticMart?",
        "ground_truth_answer": "AgenticMart offers a 30-day return policy for most electronics, provided they are in their original packaging with all accessories. Opened software or digital downloads are non-returnable. A 15% restocking fee may apply for certain items.",
        "ground_truth_contexts": [
            "AgenticMart Returns Policy: Electronics",
            "AgenticMart Terms & Conditions: Digital Goods"
        ]
    }
]

# --- 2. Simulate Agentic RAG System Outputs ---
# In a real application, these would be the actual outputs from your AutoGen agents
# after processing the queries from the golden dataset.

# We'll intentionally introduce some imperfections to demonstrate evaluation.
rag_outputs = [
    {
        "question": "What are the main features of the 'Quantum Leap' gaming laptop?",
        "answer": "The Quantum Leap laptop features an Intel Core i9 processor, an NVIDIA RTX 5090 GPU, 32GB RAM, and a 17-inch 240Hz display. It also has a unique RGB keyboard and advanced cooling.", # Slight hallucination (RGB keyboard, advanced cooling not in ground truth context)
        "contexts": [
            "Quantum Leap Product Page: Features - Intel Core i9-14900HX, NVIDIA GeForce RTX 5090, 32GB DDR5, 17-inch QHD 240Hz, 2TB NVMe SSD.",
            "Gaming Laptop Reviews: General Features"
        ] # Added irrelevant context
    },
    {
        "question": "How do I troubleshoot connectivity issues with the 'AuraFlow' smart air purifier?",
        "answer": "You can fix AuraFlow connectivity by checking the power and Wi-Fi. Restarting helps. Also, ensure your phone app is updated.", # Slightly less comprehensive than ground truth
        "contexts": [
            "AuraFlow User Manual: Troubleshooting - Connectivity Issues",
            "General Smart Device Troubleshooting Tips"
        ] # Missing a key context (factory reset)
    },
    {
        "question": "What is the return policy for electronics purchased from AgenticMart?",
        "answer": "AgenticMart allows returns for electronics within 30 days. Make sure the item is in its original condition. Some items might have a restocking fee.", # Good answer, but slightly less detail on non-returnable items
        "contexts": [
            "AgenticMart Returns Policy: Electronics"
        ] # Missing a relevant context (digital goods)
    }
]

# --- 3. Combine Golden Data and RAG Outputs for Evaluation ---
# Ragas expects a specific format: a Hugging Face `Dataset` object.
# We need to merge the ground truth with the RAG system's actual outputs.

evaluation_data = []
for i in range(len(golden_data)):
    entry = {
        "question": golden_data[i]["question"],
        "ground_truth": [golden_data[i]["ground_truth_answer"]], # Ragas expects a list for ground_truth
        "answer": rag_outputs[i]["answer"],
        "contexts": rag_outputs[i]["contexts"]
    }
    evaluation_data.append(entry)

# Convert to Hugging Face Dataset format
dataset = Dataset.from_list(evaluation_data)

print("\n--- Dataset for Ragas Evaluation ---")
print(dataset)
print("\nExample entry:")
print(dataset[0])

# --- 4. Run Ragas Evaluation ---
# This step uses LLMs to score each metric for each entry in the dataset.
# This can take some time and consumes API credits.

print("\n--- Starting Ragas Evaluation (this may take a few minutes) ---")

# Define the metrics we want to evaluate
metrics_to_evaluate = [
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_similarity
]

# Perform the evaluation
# If you encounter issues with LLM providers, ensure your API keys are correctly set
# and that you have sufficient credits.
# For local LLMs, you might need to configure Ragas with a specific LLM provider.

# Note: For a quick test without actual LLM calls, you can mock the Ragas LLM provider
# or use a very small, local model if configured. For this example, we assume
# a working OpenAI API key is available for the default Ragas LLM provider.

# If you don't have an OpenAI key, you can comment out the evaluate call and
# manually inspect the 'dataset' variable to understand the input format.

try:
    result = evaluate(
        dataset,
        metrics=metrics_to_evaluate,
        # You can specify a different LLM provider here if needed, e.g.,
        # llm=OpenAIChat(model_name="gpt-4o"),
        # embeddings=OpenAIEmbeddings(model_name="text-embedding-3-large")
    )
    print("\n--- Ragas Evaluation Results ---")
    print(result)
    print("\n--- Detailed Scores Per Metric ---")
    print(result.to_pandas())

except Exception as e:
    print(f"\nError during Ragas evaluation: {e}")
    print("Please ensure your OPENAI_API_KEY is correctly set and you have internet connectivity.")
    print("You might also need to install `tiktoken` for OpenAI models.")

print("\n--- Evaluation Complete ---")


### Interpreting the Evaluation Output and Performance Trade-offs

The Ragas evaluation provides a comprehensive score for each metric, typically ranging from 0 to 1, where 1 indicates perfect performance. Let's break down what these scores mean and how to interpret them in the context of our eCommerce RAG system:

*   **Faithfulness:** A low faithfulness score (e.g., below 0.8) indicates that your RAG system is likely **hallucinating** – generating information not supported by the retrieved context. For an eCommerce agent, this is critical; a hallucinated product feature or price can lead to customer dissatisfaction and returns.
*   **Answer Relevancy:** A low score here suggests the agent's answers are often **off-topic** or don't directly address the user's query. In eCommerce, this means customers aren't getting the information they need, leading to frustration and potentially lost sales.
*   **Context Recall:** If this score is low, it implies your retrieval mechanism is **missing crucial information** that's necessary to fully answer the query. For example, if a customer asks about product dimensions and the retrieved context only contains color options, context recall would be low.
*   **Context Precision:** A low context precision score means your retrieval is bringing back **too much irrelevant information** alongside the useful bits. This can confuse the LLM during generation, potentially leading to less focused or even incorrect answers. It also increases token usage and latency.
*   **Answer Similarity:** This metric directly compares the generated answer to the ground truth. A low score here means the agent's answer, even if relevant and faithful, isn't as accurate or complete as the ideal answer. This is a direct measure of overall correctness against your golden standard.

By examining these scores, you can pinpoint specific weaknesses in your agentic RAG pipeline:

*   **Low Faithfulness + High Context Recall:** The retriever is finding the right info, but the generator is hallucinating. Focus on prompt engineering for the LLM or fine-tuning.
*   **Low Context Recall:** The retriever isn't finding all necessary information. Improve your indexing, chunking strategy, or retrieval algorithms (e.g., hybrid search, re-ranking).
*   **Low Context Precision:** The retriever is bringing back too much noise. Refine your search queries, implement better filtering, or use a more precise re-ranker.
*   **Low Answer Relevancy:** The agent might be misinterpreting the query or getting sidetracked. Improve query understanding or agentic planning.

### Performance Trade-offs and Use Cases

**Trade-offs:**

1.  **Cost and Latency of LLM-as-a-Judge:** Using powerful LLMs for evaluation (as Ragas does) incurs API costs and can be slow, especially for large datasets. This is a trade-off for the high quality and nuanced assessment they provide.
2.  **Golden Dataset Creation:** Building and maintaining a high-quality golden dataset is labor-intensive and expensive. It requires human expertise to define ground truth, which is a significant upfront and ongoing investment.
3.  **Metric Selection:** Not all metrics are equally important for every use case. For a legal RAG, faithfulness is paramount. For a creative writing assistant, answer similarity might be less critical than fluency. Choose metrics that align with your application's goals.

**Typical Use Cases:**

*   **Continuous Integration/Continuous Deployment (CI/CD):** Integrate RAG evaluation into your CI/CD pipeline. Automatically run evaluations against a golden dataset whenever code changes are pushed. If scores drop below a threshold, block the deployment.
*   **A/B Testing:** When experimenting with new retrieval strategies, LLM models, or prompt templates, use evaluation metrics to objectively compare the performance of different versions.
*   **Production Monitoring:** Periodically run evaluations on a sample of live queries to detect performance degradation over time, which could indicate concept drift or issues with data freshness.
*   **Debugging and Improvement:** Use detailed evaluation results to diagnose specific issues and guide your efforts in improving the RAG pipeline components.


### Resources

*   **Ragas Documentation:** The official documentation for Ragas, a leading framework for RAG evaluation. [https://docs.ragas.io/](https://docs.ragas.io/)
*   **Hugging Face `datasets` Library:** Learn more about creating and managing datasets for machine learning. [https://huggingface.co/docs/datasets/index](https://huggingface.co/docs/datasets/index)
*   **TruLens:** Another powerful RAG evaluation and observability framework. [https://www.trulens.org/](https://www.trulens.org/)
*   **DeepEval:** An open-source LLM evaluation framework. [https://github.com/confident-ai/deepeval](https://github.com/confident-ai/deepeval)
*   **OpenAI API Documentation:** For understanding LLM usage and API keys. [https://platform.openai.com/docs/overview](https://platform.openai.com/docs/overview)
